In [6]:
print(1+5)

6


In [2]:
# import time
# time.sleep(5)

In [4]:
spark

In [8]:
df = spark.sql("""
    SELECT
        b.tour_id,
        SUM(b.tickets) AS total_tickets_sold
    FROM production.dwh.fact_booking b
    INNER JOIN production.dwh.dim_status s ON b.status_id = s.status_id
    WHERE s.status_name = 'active'
      AND YEAR(b.date_of_checkout_local) = 2024
    GROUP BY b.tour_id
    ORDER BY total_tickets_sold DESC
    LIMIT 10
""")

In [9]:
display(df)

,tour_id,total_tickets_sold
0,62214,362286
1,50027,311630
2,193940,278280
3,53791,271705
4,73250,249158
5,195566,185401
6,153354,178258
7,145779,172828
8,419804,172374
9,416276,172361


In [10]:
print(df.count())

10


In [6]:
print(df.count())

10


In [7]:
print(df.count())

10


In [17]:
time.sleep(10)

In [4]:
print(3+5)

8


In [9]:
total = 0
for i in range(10**8):
    total += 1

In [10]:
print(total)

100000000


In [4]:
spark.sql("""
    SELECT
        b.tour_id,
        SUM(b.tickets) AS total_tickets_sold
    FROM production.dwh.fact_booking b
    INNER JOIN production.dwh.dim_status s ON b.status_id = s.status_id
    WHERE s.status_name = 'active'
      AND YEAR(b.date_of_checkout_local) = 2024
    GROUP BY b.tour_id
    ORDER BY total_tickets_sold DESC
    LIMIT 10
""").display()

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `display` is not supported.

In [12]:
spark.sql("DESCRIBE production.dwh.dim_tour").filter("col_name LIKE '%activ%' OR col_name LIKE '%deactiv%' OR col_name LIKE '%status%' OR col_name LIKE '%online%'").show(50, truncate=False)

+-------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------+
|col_name                                         |data_type|comment                                                                                                                         |
+-------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------+
|is_online                                        |boolean  |A boolean flag indicating whether the tour is currently available for booking online. (AI-generated field)                      |
|status                                           |string   |The current status of the tour, which may include values such as active, inactive, or pending. (AI-generated field)             |
|gyg_status                                  

In [13]:
# Tour activity on the GYG website for each day of 2025.
# dim_tour columns used: date_of_first_online (activated), last_online_date + is_online (deactivated).
tour_active_2025 = spark.sql("""
    WITH date_spine AS (
        SELECT explode(sequence(
            DATE '2025-01-01',
            DATE '2025-12-31',
            INTERVAL 1 DAY
        )) AS dt
    ),
    tours AS (
        SELECT
            tour_id,
            CAST(date_of_first_online AS DATE) AS activated_date,
            CASE WHEN is_online = FALSE THEN CAST(last_online_date AS DATE) ELSE NULL END AS deactivated_date
        FROM production.dwh.dim_tour
        WHERE date_of_first_online IS NOT NULL
    )
    SELECT
        d.dt       AS date,
        t.tour_id,
        CASE
            WHEN d.dt >= t.activated_date
             AND (t.deactivated_date IS NULL OR d.dt <= t.deactivated_date)
            THEN 1 ELSE 0
        END        AS is_active_on_website
    FROM date_spine d
    CROSS JOIN tours t
    ORDER BY t.tour_id, d.dt
""")

print(f"Shape: {tour_active_2025.count():,} rows x {len(tour_active_2025.columns)} cols")
tour_active_2025.show(10)

Shape: 204,685,430 rows x 3 cols


+----------+-------+--------------------+
|      date|tour_id|is_active_on_website|
+----------+-------+--------------------+
|2025-01-01|      1|                   1|
|2025-01-02|      1|                   1|
|2025-01-03|      1|                   1|
|2025-01-04|      1|                   1|
|2025-01-05|      1|                   1|
|2025-01-06|      1|                   1|
|2025-01-07|      1|                   1|
|2025-01-08|      1|                   1|
|2025-01-09|      1|                   1|
|2025-01-10|      1|                   1|
+----------+-------+--------------------+
only showing top 10 rows



In [15]:
from pyspark.sql import functions as F

daily_availability = (
    tour_active_2025
    .groupBy("date")
    .agg(
        F.sum("is_active_on_website").alias("active_tours"),
        F.count("tour_id").alias("total_tours"),
    )
    .withColumn("pct_active", F.round(F.col("active_tours") / F.col("total_tours") * 100, 2))
    .orderBy("date")
)
daily_availability.show(20)

+----------+------------+-----------+----------+
|      date|active_tours|total_tours|pct_active|
+----------+------------+-----------+----------+
|2025-01-01|      140303|     560782|     25.02|
|2025-01-02|      140244|     560782|     25.01|
|2025-01-03|      140047|     560782|     24.97|
|2025-01-04|      139881|     560782|     24.94|
|2025-01-05|      139756|     560782|     24.92|
|2025-01-06|      139719|     560782|     24.92|
|2025-01-07|      139621|     560782|      24.9|
|2025-01-08|      139546|     560782|     24.88|
|2025-01-09|      139470|     560782|     24.87|
|2025-01-10|      139424|     560782|     24.86|
|2025-01-11|      139349|     560782|     24.85|
|2025-01-12|      139297|     560782|     24.84|
|2025-01-13|      139328|     560782|     24.85|
|2025-01-14|      139246|     560782|     24.83|
|2025-01-15|      139245|     560782|     24.83|
|2025-01-16|      139105|     560782|     24.81|
|2025-01-17|      139062|     560782|      24.8|
|2025-01-18|      13

In [17]:
from pyspark.sql import functions as F

# Re-build tours with full timestamps (not just dates)
tours_ts = spark.sql("""
    SELECT
        tour_id,
        date_of_first_online AS activated_ts,
        CASE WHEN is_online = FALSE THEN last_online_date ELSE NULL END AS deactivated_ts
    FROM production.dwh.dim_tour
    WHERE date_of_first_online IS NOT NULL
""")

date_spine = spark.sql("""
    SELECT explode(sequence(
        DATE '2025-01-01',
        DATE '2025-12-31',
        INTERVAL 1 DAY
    )) AS dt
""")

# For each (tour, day): compute seconds of overlap with [day_start, day_end]
tour_daily_pct = (
    date_spine.crossJoin(tours_ts)
    .withColumn("day_start", F.col("dt").cast("timestamp"))
    .withColumn("day_end",   (F.col("dt") + F.expr("INTERVAL 1 DAY")).cast("timestamp"))
    .withColumn("active_from", F.greatest("day_start", "activated_ts"))
    .withColumn("active_to",   F.least("day_end", F.coalesce("deactivated_ts", "day_end")))
    .withColumn("active_seconds", F.greatest(
        F.lit(0),
        (F.unix_timestamp("active_to") - F.unix_timestamp("active_from"))
    ))
    .withColumn("pct_of_day", F.round(F.col("active_seconds") / 86400 * 100, 2))
    .select("dt", "tour_id", "active_seconds", "pct_of_day")
    .orderBy("tour_id", "dt")
)

tour_daily_pct.show(20)

+----------+-------+--------------+----------+
|        dt|tour_id|active_seconds|pct_of_day|
+----------+-------+--------------+----------+
|2025-01-01|      1|         86400|     100.0|
|2025-01-02|      1|         86400|     100.0|
|2025-01-03|      1|         86400|     100.0|
|2025-01-04|      1|         86400|     100.0|
|2025-01-05|      1|         86400|     100.0|
|2025-01-06|      1|         86400|     100.0|
|2025-01-07|      1|         86400|     100.0|
|2025-01-08|      1|         86400|     100.0|
|2025-01-09|      1|         86400|     100.0|
|2025-01-10|      1|         86400|     100.0|
|2025-01-11|      1|         86400|     100.0|
|2025-01-12|      1|         86400|     100.0|
|2025-01-13|      1|         86400|     100.0|
|2025-01-14|      1|         86400|     100.0|
|2025-01-15|      1|         86400|     100.0|
|2025-01-16|      1|         86400|     100.0|
|2025-01-17|      1|         86400|     100.0|
|2025-01-18|      1|         86400|     100.0|
|2025-01-19| 

In [18]:
spark.sql("SHOW TABLES IN production.dwh LIKE '*option*'").show(20, truncate=False)

+--------+---------------------------------------------+-----------+
|database|tableName                                    |isTemporary|
+--------+---------------------------------------------+-----------+
|dwh     |dim_tour_option                              |false      |
|dwh     |dim_tour_option_addon                        |false      |
|dwh     |dim_tour_option_completeness                 |false      |
|dwh     |dim_tour_option_conduction_language          |false      |
|dwh     |dim_tour_option_donations                    |false      |
|dwh     |dim_tour_option_extras                       |false      |
|dwh     |dim_tour_option_history                      |false      |
|dwh     |dim_tour_option_low_inventory                |false      |
|dwh     |dim_tour_option_metrics                      |false      |
|dwh     |dim_tour_option_pricing                      |false      |
|dwh     |dim_tour_options_with_competitiveness_metrics|false      |
|dwh     |fact_tour_option_change 

In [19]:
spark.sql("DESCRIBE production.dwh.dim_tour_option").filter(
    "col_name LIKE '%activ%' OR col_name LIKE '%deactiv%' OR col_name LIKE '%online%' OR col_name LIKE '%option_id%' OR col_name LIKE '%tour_id%' OR col_name LIKE '%status%'"
).show(30, truncate=False)

+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------+
|col_name               |data_type|comment                                                                                                                       |
+-----------------------+---------+------------------------------------------------------------------------------------------------------------------------------+
|tour_option_id         |bigint   |A unique identifier for each tour option, used as the primary key in the table. (AI-generated field)                          |
|tour_id                |bigint   |A unique identifier for the tour associated with the tour option, linking it to the broader tour details. (AI-generated field)|
|status                 |string   |The current status of the tour option, indicating its availability or any restrictions. (AI-generated field)                  |
|deactivated_user     

In [20]:
spark.sql("DESCRIBE production.dwh.dim_tour_option").filter(
    "col_name LIKE '%deactiv%' OR col_name LIKE '%last%' OR col_name LIKE '%end%' OR col_name LIKE '%offline%'"
).show(20, truncate=False)

+----------------+---------+-----------------------------------------------------------------------------------------------+
|col_name        |data_type|comment                                                                                        |
+----------------+---------+-----------------------------------------------------------------------------------------------+
|deactivated_user|string   |The identifier of the user who deactivated the tour option, if applicable. (AI-generated field)|
+----------------+---------+-----------------------------------------------------------------------------------------------+



In [21]:
spark.sql("DESCRIBE production.dwh.dim_tour_option_history").filter(
    "col_name LIKE '%activ%' OR col_name LIKE '%deactiv%' OR col_name LIKE '%option_id%' OR col_name LIKE '%status%' OR col_name LIKE '%date%' OR col_name LIKE '%time%'"
).show(30, truncate=False)

+------------------------+---------+----------------------------------------------------------------------------------------------------------------------------------+
|col_name                |data_type|comment                                                                                                                           |
+------------------------+---------+----------------------------------------------------------------------------------------------------------------------------------+
|tour_option_id          |bigint   |A unique identifier for each tour option, used to distinguish between different options available for a tour. (AI-generated field)|
|status                  |string   |The current status of the tour option, indicating whether it is active, inactive, or in another state. (AI-generated field)       |
|timezone_id             |string   |The identifier for the timezone in which the tour option operates, used for scheduling and timing. (AI-generated field)     

In [12]:
from pyspark.sql import functions as F

# SCD2 history: each row covers [update_timestamp, update_timestamp_next) with a given status
# status = 'active' means the option was bookable during that window
options_ts = spark.sql("""
    SELECT
        tour_option_id,
        status,
        update_timestamp      AS active_from,
        update_timestamp_next AS active_to     -- NULL means currently valid row
    FROM production.dwh.dim_tour_option_history
    WHERE status = 'active'
""")

date_spine = spark.sql("""
    SELECT explode(sequence(
        DATE '2025-01-01',
        DATE '2025-12-31',
        INTERVAL 1 DAY
    )) AS dt
""")

option_daily_pct = (
    date_spine.crossJoin(options_ts)
    .withColumn("day_start", F.col("dt").cast("timestamp"))
    .withColumn("day_end",   (F.col("dt") + F.expr("INTERVAL 1 DAY")).cast("timestamp"))
    # clip active window to the day
    .withColumn("overlap_from", F.greatest("day_start", "active_from"))
    .withColumn("overlap_to",   F.least("day_end", F.coalesce("active_to", "day_end")))
    # only keep rows where the window actually overlaps this day
    .filter(F.col("overlap_from") < F.col("overlap_to"))
    # a single option may have multiple active windows in a day — sum them up
    .groupBy("dt", "tour_option_id")
    .agg(F.sum(
        F.unix_timestamp("overlap_to") - F.unix_timestamp("overlap_from")
    ).alias("active_seconds"))
    .withColumn("pct_of_day", F.round(F.col("active_seconds") / 86400 * 100, 2))
    .orderBy("tour_option_id", "dt")
)

option_daily_pct.show(20)

+----------+--------------+--------------+----------+
|        dt|tour_option_id|active_seconds|pct_of_day|
+----------+--------------+--------------+----------+
|2025-01-01|             9|         86400|     100.0|
|2025-01-02|             9|         86400|     100.0|
|2025-01-03|             9|         86400|     100.0|
|2025-01-04|             9|         86400|     100.0|
|2025-01-05|             9|         86400|     100.0|
|2025-01-06|             9|         86400|     100.0|
|2025-01-07|             9|         86400|     100.0|
|2025-01-08|             9|         86400|     100.0|
|2025-01-09|             9|         86400|     100.0|
|2025-01-10|             9|         86400|     100.0|
|2025-01-11|             9|         86400|     100.0|
|2025-01-12|             9|         86400|     100.0|
|2025-01-13|             9|         86400|     100.0|
|2025-01-14|             9|         86400|     100.0|
|2025-01-15|             9|         86400|     100.0|
|2025-01-16|             9| 

In [24]:
spark.sql("DESCRIBE production.dwh.fact_booking").filter(
    "col_name LIKE '%date%' OR col_name LIKE '%tour_id%' OR col_name LIKE '%ticket%' OR col_name LIKE '%travel%' OR col_name LIKE '%departure%'"
).show(30, truncate=False)

+---------------------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                               |data_type    |comment                                                                                                                                                                      |
+---------------------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date_of_checkout                       |timestamp    |The timestamp when the customer completed the checkout process, in Europe/Berlin timezone. (AI-generated field)                                                              |
|tour_id                                |bigint       |The unique identifier for

In [25]:
booking_dynamics = spark.sql("""
WITH top_tours AS (
    SELECT tour_id
    FROM production.dwh.fact_booking b
    JOIN production.dwh.dim_status s ON b.status_id = s.status_id
    WHERE s.status_name = 'active'
      AND YEAR(b.date_of_checkout_local) = 2024
    GROUP BY tour_id
    ORDER BY SUM(tickets) DESC
    LIMIT 10
),
days_before AS (
    SELECT
        b.tour_id,
        DATEDIFF(CAST(b.date_of_travel AS DATE), CAST(b.date_of_checkout_local AS DATE)) AS days_before_tour,
        b.tickets
    FROM production.dwh.fact_booking b
    JOIN production.dwh.dim_status s ON b.status_id = s.status_id
    JOIN top_tours t ON b.tour_id = t.tour_id
    WHERE s.status_name = 'active'
      AND YEAR(b.date_of_checkout_local) = 2024
      AND DATEDIFF(CAST(b.date_of_travel AS DATE), CAST(b.date_of_checkout_local AS DATE)) BETWEEN 1 AND 30
)
SELECT
    tour_id,
    days_before_tour,
    SUM(tickets) AS tickets_sold
FROM days_before
GROUP BY tour_id, days_before_tour
ORDER BY tour_id, days_before_tour
""")

booking_dynamics.show(30)

+-------+----------------+------------+
|tour_id|days_before_tour|tickets_sold|
+-------+----------------+------------+
|  50027|               1|        8379|
|  50027|               2|       11755|
|  50027|               3|       13631|
|  50027|               4|       15374|
|  50027|               5|       17610|
|  50027|               6|       18475|
|  50027|               7|       18366|
|  50027|               8|       15630|
|  50027|               9|       14088|
|  50027|              10|       12483|
|  50027|              11|       11115|
|  50027|              12|        9058|
|  50027|              13|        8544|
|  50027|              14|        7861|
|  50027|              15|        7254|
|  50027|              16|        6448|
|  50027|              17|        5585|
|  50027|              18|        5475|
|  50027|              19|        5166|
|  50027|              20|        4480|
|  50027|              21|        4395|
|  50027|              22|        4319|


In [26]:
booking_dynamics_v2 = spark.sql("""
WITH top_tours AS (
    SELECT tour_id
    FROM production.dwh.fact_booking
    WHERE YEAR(date_of_checkout_local) = 2024
    GROUP BY tour_id
    ORDER BY SUM(tickets) DESC
    LIMIT 10
),
days_before AS (
    SELECT
        b.tour_id,
        DATEDIFF(CAST(b.date_of_travel AS DATE), CAST(b.date_of_checkout_local AS DATE)) AS days_before_tour,
        b.tickets
    FROM production.dwh.fact_booking b
    JOIN top_tours t ON b.tour_id = t.tour_id
    WHERE YEAR(b.date_of_checkout_local) = 2024
      AND DATEDIFF(CAST(b.date_of_travel AS DATE), CAST(b.date_of_checkout_local AS DATE)) BETWEEN 1 AND 30
)
SELECT
    tour_id,
    days_before_tour,
    SUM(tickets) AS tickets_sold
FROM days_before
GROUP BY tour_id, days_before_tour
ORDER BY tour_id, days_before_tour
""")

booking_dynamics_v2.show(30)

+-------+----------------+------------+
|tour_id|days_before_tour|tickets_sold|
+-------+----------------+------------+
|  50027|               1|        8491|
|  50027|               2|       12155|
|  50027|               3|       14355|
|  50027|               4|       16282|
|  50027|               5|       18693|
|  50027|               6|       19628|
|  50027|               7|       19614|
|  50027|               8|       16708|
|  50027|               9|       15174|
|  50027|              10|       13487|
|  50027|              11|       12099|
|  50027|              12|        9859|
|  50027|              13|        9299|
|  50027|              14|        8600|
|  50027|              15|        7907|
|  50027|              16|        7097|
|  50027|              17|        6112|
|  50027|              18|        5891|
|  50027|              19|        5624|
|  50027|              20|        4886|
|  50027|              21|        4831|
|  50027|              22|        4734|


In [27]:
booking_snapshot = spark.sql("""
WITH top_tours AS (
    SELECT tour_id
    FROM production.dwh.fact_booking
    WHERE YEAR(date_of_checkout_local) = 2024
    GROUP BY tour_id
    ORDER BY SUM(tickets) DESC
    LIMIT 10
),
-- one row per (tour, booking) with the key dates
base AS (
    SELECT
        b.tour_id,
        b.tickets,
        CAST(b.date_of_travel         AS DATE) AS travel_date,
        CAST(b.date_of_checkout_local  AS DATE) AS checkout_date,
        CAST(b.date_of_cancelation     AS DATE) AS cancel_date
    FROM production.dwh.fact_booking b
    JOIN top_tours t ON b.tour_id = t.tour_id
    WHERE YEAR(b.date_of_checkout_local) = 2024
),
-- cross with 1..30 days-before values
snapshots AS (
    SELECT
        b.tour_id,
        n.days_before,
        b.tickets,
        b.travel_date,
        -- snapshot date = travel_date minus N days
        DATE_SUB(b.travel_date, n.days_before) AS snapshot_date,
        b.checkout_date,
        b.cancel_date
    FROM base b
    CROSS JOIN (
        SELECT explode(sequence(1, 30)) AS days_before
    ) n
    WHERE
        -- booking existed at snapshot time
        b.checkout_date <= DATE_SUB(b.travel_date, n.days_before)
        -- not yet cancelled at snapshot time
        AND (b.cancel_date IS NULL OR b.cancel_date > DATE_SUB(b.travel_date, n.days_before))
)
SELECT
    tour_id,
    days_before,
    SUM(tickets) AS active_tickets_at_snapshot
FROM snapshots
GROUP BY tour_id, days_before
ORDER BY tour_id, days_before
""")

booking_snapshot.show(30)

+-------+-----------+--------------------------+
|tour_id|days_before|active_tickets_at_snapshot|
+-------+-----------+--------------------------+
|  50027|          1|                    308979|
|  50027|          2|                    300766|
|  50027|          3|                    290276|
|  50027|          4|                    278027|
|  50027|          5|                    263415|
|  50027|          6|                    246082|
|  50027|          7|                    227808|
|  50027|          8|                    209504|
|  50027|          9|                    193828|
|  50027|         10|                    179642|
|  50027|         11|                    167095|
|  50027|         12|                    155886|
|  50027|         13|                    146777|
|  50027|         14|                    138128|
|  50027|         15|                    130103|
|  50027|         16|                    122686|
|  50027|         17|                    116039|
|  50027|         18

In [28]:
spark.sql("SHOW TABLES IN production.dwh LIKE '*booking*status*' OR LIKE '*status*booking*' OR LIKE '*booking*history*' OR LIKE '*booking*change*' OR LIKE '*booking*log*'").show(30, truncate=False)

ParseException: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'OR'. SQLSTATE: 42601 (line 1, pos 54)

== SQL ==
SHOW TABLES IN production.dwh LIKE '*booking*status*' OR LIKE '*status*booking*' OR LIKE '*booking*history*' OR LIKE '*booking*change*' OR LIKE '*booking*log*'
------------------------------------------------------^^^


JVM stacktrace:
org.apache.spark.sql.catalyst.parser.ParseException
	at org.apache.spark.sql.catalyst.parser.ParseException.withCommand(parsers.scala:429)
	at org.apache.spark.sql.catalyst.parser.AbstractParser.parse(parsers.scala:117)
	at org.apache.spark.sql.execution.SparkSqlParser.parse(SparkSqlParser.scala:140)
	at org.apache.spark.sql.catalyst.parser.AbstractSqlParser.parsePlan(AbstractSqlParser.scala:106)
	at com.databricks.sql.parser.DatabricksSqlParser.$anonfun$parsePlan$1(DatabricksSqlParser.scala:80)
	at com.databricks.sql.parser.DatabricksSqlParser.parse(DatabricksSqlParser.scala:101)
	at com.databricks.sql.parser.DatabricksSqlParser.parsePlan(Data

In [29]:
spark.sql("SHOW TABLES IN production.dwh LIKE '*booking*'").show(50, truncate=False)

+--------+---------------------------------+-----------+
|database|tableName                        |isTemporary|
+--------+---------------------------------+-----------+
|dwh     |agg_booking_gwc                  |false      |
|dwh     |agg_bookings_checkout_date_daily |false      |
|dwh     |agg_bookings_checkout_date_hourly|false      |
|dwh     |agg_bookings_tour_checkout_daily |false      |
|dwh     |agg_bookings_tour_travel_daily   |false      |
|dwh     |agg_bookings_travel_date_daily   |false      |
|dwh     |bookings_change_monitoring       |false      |
|dwh     |dim_booking_addon_details        |false      |
|dwh     |dim_booking_cancellation_category|false      |
|dwh     |dim_booking_cancellation_reason  |false      |
|dwh     |dim_booking_cancellation_user    |false      |
|dwh     |dim_booking_cohort               |false      |
|dwh     |dim_booking_geo_mapping          |false      |
|dwh     |dim_booking_history              |false      |
|dwh     |dim_booking_language 

In [30]:
spark.sql("DESCRIBE production.dwh.fact_booking_snapshot").show(50, truncate=False)
spark.sql("DESCRIBE production.dwh.dim_booking_history").filter(
    "col_name LIKE '%status%' OR col_name LIKE '%ticket%' OR col_name LIKE '%date%' OR col_name LIKE '%tour_id%' OR col_name LIKE '%booking%id%'"
).show(30, truncate=False)

+----------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                                |data_type    |comment                                                                                                                                                 |
+----------------------------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|snapshot_timestamp                      |timestamp    |The 'snapshot_timestamp' column records the exact time when the snapshot of the booking data was taken. (AI-generated field)                            |
|booking_id                              |bigint       |The 'booking_id' column uniquely identifies each booking within the system. (AI-generated field)        

+---------------------------------+-------------+--------------------------------------------------------------------------------------------+
|col_name                         |data_type    |comment                                                                                     |
+---------------------------------+-------------+--------------------------------------------------------------------------------------------+
|booking_id                       |bigint       |A unique identifier for each booking. (AI-generated field)                                  |
|tour_start_datetime_utc_timestamp|bigint       |The start date and time of the tour in UTC timestamp format. (AI-generated field)           |
|all_travel_dates_utc             |array<string>|An array of all travel dates in UTC format associated with the booking. (AI-generated field)|
+---------------------------------+-------------+--------------------------------------------------------------------------------------------+

In [31]:
spark.sql("DESCRIBE production.dwh.fact_booking_snapshot").filter(
    "col_name IN ('tickets', 'status_id', 'snapshot_timestamp') OR col_name LIKE '%status%' OR col_name LIKE '%ticket%'"
).show(20, truncate=False)

+------------------+---------+----------------------------------------------------------------------------------------------------------------------------+
|col_name          |data_type|comment                                                                                                                     |
+------------------+---------+----------------------------------------------------------------------------------------------------------------------------+
|snapshot_timestamp|timestamp|The 'snapshot_timestamp' column records the exact time when the snapshot of the booking data was taken. (AI-generated field)|
|tickets           |bigint   |The 'tickets' column indicates the number of tickets booked for the tour or activity. (AI-generated field)                  |
|deal_tickets      |bigint   |The 'deal_tickets' column records the number of tickets associated with a promotional deal. (AI-generated field)            |
|status_id         |int      |The 'status_id' column indicates t

In [32]:
spark.sql("""
    SELECT DISTINCT CAST(snapshot_timestamp AS DATE) AS snapshot_date
    FROM production.dwh.fact_booking_snapshot
    ORDER BY snapshot_date DESC
    LIMIT 10
""").show()

+-------------+
|snapshot_date|
+-------------+
|   2026-06-11|
+-------------+



In [33]:
spark.sql("DESCRIBE production.dwh.dim_booking_history").show(50, truncate=False)

+-----------------------------------+-------------+----------------------------------------------------------------------------------------------+
|col_name                           |data_type    |comment                                                                                       |
+-----------------------------------+-------------+----------------------------------------------------------------------------------------------+
|supplier_total_price               |double       |The total price charged by the supplier for the booking. (AI-generated field)                 |
|booking_id                         |bigint       |A unique identifier for each booking. (AI-generated field)                                    |
|reseller_total_price               |double       |The total price charged by the reseller for the booking. (AI-generated field)                 |
|customer_total_price               |double       |The total price paid by the customer for the booking. (AI-generated

In [34]:
for tbl in ['fact_booking_v2', 'fact_booking_extended']:
    print(f"\n=== {tbl} ===")
    spark.sql(f"DESCRIBE production.dwh.{tbl}").filter(
        "col_name LIKE '%status%' OR col_name LIKE '%ticket%' OR col_name LIKE '%date%' OR col_name LIKE '%valid%' OR col_name LIKE '%update%'"
    ).show(20, truncate=False)


=== fact_booking_v2 ===


+---------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|col_name                   |data_type    |comment                                                                                                                                                                      |
+---------------------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date_of_checkout           |timestamp    |The timestamp when the customer completed the checkout process, in Europe/Berlin timezone. (AI-generated field)                                                              |
|status_id                  |int          |The unique identifier for the current status of the booking (e.g., confirmed, cancele

+--------+---------+-------+
|col_name|data_type|comment|
+--------+---------+-------+
+--------+---------+-------+



In [35]:
for tbl in ['bookings_change_monitoring', 'fact_booking_cancellation_request']:
    print(f"\n=== {tbl} ===")
    spark.sql(f"DESCRIBE production.dwh.{tbl}").show(20, truncate=False)


=== bookings_change_monitoring ===


+-----------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------+
|col_name               |data_type    |comment                                                                                                                               |
+-----------------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------+
|table_name             |string       |The name of the source table being monitored for changes, typically 'fact_booking'. (AI-generated field)                              |
|pk                     |string       |The name of the primary key column in the source table, typically 'booking_id'. (AI-generated field)                                  |
|pk_value               |bigint       |The actual value of the primary key (booking_id) for the record where a change was det

+------------------------------+---------+-----------------------------------------------------------------------------------------------------------------+
|col_name                      |data_type|comment                                                                                                          |
+------------------------------+---------+-----------------------------------------------------------------------------------------------------------------+
|booking_cancelation_request_id|bigint   |A unique identifier for each booking cancellation request. (AI-generated field)                                  |
|booking_cancelation_reason_id |bigint   |An identifier linking to the reason for the booking cancellation. (AI-generated field)                           |
|booking_cancelation_user_id   |int      |An identifier for the user who requested the booking cancellation. (AI-generated field)                          |
|booking_id                    |bigint   |The unique ident

In [36]:
booking_snapshot_v2 = spark.sql("""
WITH top_tours AS (
    SELECT tour_id
    FROM production.dwh.fact_booking
    WHERE YEAR(date_of_checkout_local) = 2024
    GROUP BY tour_id
    ORDER BY SUM(tickets) DESC
    LIMIT 10
),
base AS (
    SELECT
        b.booking_id,
        b.tour_id,
        b.tickets,
        CAST(b.date_of_travel        AS DATE) AS travel_date,
        CAST(b.date_of_checkout_local AS DATE) AS checkout_date,
        -- use cancellation request table as point-in-time source for cancellations
        CAST(cr.date_of_cancelation   AS DATE) AS cancel_date
    FROM production.dwh.fact_booking b
    JOIN top_tours t ON b.tour_id = t.tour_id
    LEFT JOIN production.dwh.fact_booking_cancellation_request cr ON b.booking_id = cr.booking_id
    WHERE YEAR(b.date_of_checkout_local) = 2024
),
snapshots AS (
    SELECT
        b.tour_id,
        n.days_before,
        b.tickets
    FROM base b
    CROSS JOIN (SELECT explode(sequence(1, 30)) AS days_before) n
    WHERE
        -- booking existed at snapshot time
        b.checkout_date <= DATE_SUB(b.travel_date, n.days_before)
        -- not yet cancelled at snapshot time
        AND (b.cancel_date IS NULL OR b.cancel_date > DATE_SUB(b.travel_date, n.days_before))
)
SELECT
    tour_id,
    days_before,
    SUM(tickets) AS active_tickets_at_snapshot
FROM snapshots
GROUP BY tour_id, days_before
ORDER BY tour_id, days_before
""")

booking_snapshot_v2.show(30)

+-------+-----------+--------------------------+
|tour_id|days_before|active_tickets_at_snapshot|
+-------+-----------+--------------------------+
|  50027|          1|                    308977|
|  50027|          2|                    300766|
|  50027|          3|                    290276|
|  50027|          4|                    278027|
|  50027|          5|                    263415|
|  50027|          6|                    246082|
|  50027|          7|                    227808|
|  50027|          8|                    209504|
|  50027|          9|                    193828|
|  50027|         10|                    179642|
|  50027|         11|                    167095|
|  50027|         12|                    155886|
|  50027|         13|                    146777|
|  50027|         14|                    138128|
|  50027|         15|                    130103|
|  50027|         16|                    122686|
|  50027|         17|                    116039|
|  50027|         18

In [13]:
partial_availability = option_daily_pct.filter(
    (F.col("pct_of_day") > 0) & (F.col("pct_of_day") < 100)
)
print(partial_availability.count())
partial_availability.show(20)

842918


+----------+--------------+--------------+----------+
|        dt|tour_option_id|active_seconds|pct_of_day|
+----------+--------------+--------------+----------+
|2025-03-30|             9|         82800|     95.83|
|2025-03-30|            11|         82800|     95.83|
|2025-03-30|            13|         82800|     95.83|
|2025-06-04|            13|         53442|     61.85|
|2025-03-30|            15|         82800|     95.83|
|2025-03-30|            25|         82800|     95.83|
|2025-06-03|            25|         71567|     82.83|
|2025-06-27|            25|         34866|     40.35|
|2025-03-30|            29|         82800|     95.83|
|2025-03-30|            33|         82800|     95.83|
|2025-03-30|            35|         82800|     95.83|
|2025-03-30|            37|         82800|     95.83|
|2025-03-30|            39|         82800|     95.83|
|2025-03-30|            41|         82800|     95.83|
|2025-03-30|            47|         82800|     95.83|
|2025-03-30|           109| 